In [1]:
# Đọc dữ liệu
import json
import pandas as pd

with open("../output/products_raw.json", "r", encoding="utf-8") as file:
    products = json.load(file)

df = pd.DataFrame(products)

print("Tổng sản phẩm:", len(products))

Tổng sản phẩm: 1146


In [2]:
# Kiểm tra số lượng giá trị rỗng theo từng trường
for column in df.columns:
    empty_count = 0

    for value in df[column]:
        if value == "" or value == []:
            empty_count += 1

    print(f"{column}: {empty_count}")

product_id: 165
product_name: 0
brand: 0
category: 0
price: 0
original_price: 410
sizes: 165
colors: 165
image_urls: 2
product_url: 0
description: 0


In [3]:
# Xóa sản phẩm có image_urls rỗng hoặc không tồn tại
products = [
    product
    for product in products
    if product.get("image_urls")
]

print("Tổng sản phẩm sau khi xóa:", len(products))

Tổng sản phẩm sau khi xóa: 1144


In [4]:
# Random product_id cho sản phẩm nào đang thiếu
import random

# Lấy tất cả product_id đang tồn tại
used_ids = set()

for product in products:

    product_id = str(product["product_id"]).strip()

    if product_id != "":
        used_ids.add(product_id)

# Sinh product_id cho các sản phẩm bị thiếu
for product in products:

    if str(product["product_id"]).strip() == "":

        while True:

            new_id = "".join(
                random.choice("0123456789")
                for _ in range(18)
            )

            if new_id not in used_ids:

                product["product_id"] = new_id
                used_ids.add(new_id)

                break

print("Done")

# Kiểm tra còn sản phẩm nào thiếu product_id không:
count = 0

for product in products:

    if str(product["product_id"]).strip() == "":
        count += 1

print("Thiếu product_id:", count)

# Kiểm tra có bị trùng product_id không:
all_ids = [
    product["product_id"]
    for product in products
]

print("Tổng ID:", len(all_ids))
print("ID duy nhất:", len(set(all_ids)))
print("Trùng:", len(all_ids) - len(set(all_ids)))

Done
Thiếu product_id: 0
Tổng ID: 1144
ID duy nhất: 1144
Trùng: 0


In [5]:
# Chuẩn hóa price và original_price
for product in products:

    # price
    if product["price"]:

        product["price"] = int(
            str(product["price"])
            .replace("₫", "")
            .replace(",", "")
            .strip()
        )

    # original_price
    if product["original_price"]:

        product["original_price"] = int(
            str(product["original_price"])
            .replace("₫", "")
            .replace(",", "")
            .strip()
        )

print("Done")

# Kiểm tra kết quả:
for product in products[:5]:
    print(product["price"], type(product["price"]))

Done
1399000 <class 'int'>
1399000 <class 'int'>
1599000 <class 'int'>
1599000 <class 'int'>
1399000 <class 'int'>


In [6]:
# Random original_price cho sản phẩm nào đang thiếu
import random

for product in products:

    if not product["original_price"]:

        product["original_price"] = (
            product["price"] +
            random.choice([360000, 700000])
        )

print("Done")

# Kiểm tra kết quả:
for product in products[:10]:
    print(
        product["price"],
        "=>",
        product["original_price"]
    )

Done
1399000 => 1759000
1399000 => 2099000
1599000 => 1959000
1599000 => 1959000
1399000 => 2099000
1399000 => 2099000
1399000 => 1759000
1399000 => 2099000
1399000 => 1759000
1399000 => 1759000


In [7]:
# Chuẩn hóa colors
for product in products:

    colors = product["colors"]

    # Xóa phần tử rỗng
    colors = [
        color.strip()
        for color in colors
        if str(color).strip() != ""
    ]

    # Nếu không còn màu nào thì gán Hỗn hợp
    if len(colors) == 0:
        colors = ["Hỗn hợp"]

    product["colors"] = colors

print("Done")

# Kiểm tra lại:
from collections import Counter

counter = Counter()

for product in products:
    counter[str(product["colors"])] += 1

print(counter.most_common())

Done
[("['Hỗn hợp']", 344), ("['Xanh']", 143), ("['Đen']", 133), ("['Nâu']", 122), ("['Trắng']", 91), ("['Hồng']", 69), ("['Xám']", 65), ("['Đỏ']", 59), ("['Kẻ']", 42), ("['Xanh lá']", 33), ("['Vàng']", 20), ("['Tím']", 17), ("['Cam']", 6)]


In [8]:
# Chuẩn hóa sizes
for product in products:

    if not product["sizes"]:

        product["sizes"] = [
            "Free size"
        ]

    else:

        product["sizes"] = [
            "Free size"
            if str(size).replace("\xa0", " ").strip() == "SP không chia size"
            else size
            for size in product["sizes"]
        ]

print("Done")

# Kiểm tra còn size "SP không chia size" không:
for product in products:

    for size in product["sizes"]:

        if "không chia size" in str(size):
            print(size)

Done


In [9]:
import json
import pandas as pd

# JSON
with open("../output/products_clean.json","w", encoding="utf-8") as file:
    json.dump(products, file, ensure_ascii=False, indent=4)

# CSV
df = pd.DataFrame(products)

df.to_csv("../output/products_clean.csv", index=False, encoding="utf-8-sig")

print("Done!")

Done!
